# Task 6 · GRPO only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 1. Install the pinned GPU training environment
After this cell, restart the runtime if Colab requests it, then rerun the clone/setup cell.
Do not install the CPU-only requirements in vendor/organizer into this GPU environment.


In [ ]:
%pip -q install -r requirements.txt
!python -m pip check
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Copy data from Drive or download directly from Hugging Face
All extraction and image reads happen on local SSD, not mounted Drive.


In [ ]:
import shutil
USE_DRIVE_DATA = True
drive_raw = Path('/content/drive/MyDrive/MMCQA/task6-data/raw')
if USE_DRIVE_DATA:
    if not drive_raw.exists():
        raise FileNotFoundError('Run the transfer notebook first, or set USE_DRIVE_DATA=False')
    shutil.copytree(drive_raw, Path('data/raw'), dirs_exist_ok=True)
else:
    subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## 3. Editable experiment settings
Edit prompts/qa.txt and YAML files, or use dotted --set overrides below.
4-bit is explicit for a memory-limited Colab GPU. Set False for native precision.
GPU smoke runs are needed before full training; they have not been executed on the CPU-only development machine.


In [ ]:
USE_4BIT = True
RUN_FULL = False
RUN_NAME = 'experiment01'  # use a new name for a fresh experiment
RUNS = Path('/content/drive/MyDrive/MMCQA/task6-runs') / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT = str(RUNS / ('sft' if RUN_FULL else 'sft-smoke'))
GRPO = str(RUNS / ('grpo' if RUN_FULL else 'grpo-smoke'))

def run(module, config, overrides):
    cmd = [sys.executable, '-m', module, '--config', config]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={value}']
    subprocess.run(cmd, check=True)

common = {'model.load_in_4bit': str(USE_4BIT).lower()}


## 4. Choose the SFT checkpoint saved on Drive
Run this notebook in a new GPU session after notebook 02. No SFT is run here.
Set SFT_ADAPTER to the final adapter path printed by notebook 02, or to an SFT checkpoint-N.
RUN_NAME above controls the GRPO output path; SFT_ADAPTER independently selects its input.
Use a full SFT run as input for a real experiment; a smoke adapter only verifies the pipeline.


In [ ]:
SFT_ADAPTER = '/content/drive/MyDrive/MMCQA/task6-runs/experiment01/sft-smoke/final'
ADAPTER_REVISION = 'null'  # local Drive adapter, no Hub revision

sft_input = Path(SFT_ADAPTER)
if not (sft_input / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'Set SFT_ADAPTER to an existing SFT adapter on Drive: {sft_input}')
if not any((sft_input / name).is_file() for name in ['adapter_model.safetensors', 'adapter_model.bin']):
    raise FileNotFoundError('SFT adapter weights are missing; wait for the SFT save to complete')
if sft_input.resolve() == (Path(GRPO) / 'final').resolve() or Path(GRPO).resolve() in sft_input.resolve().parents:
    raise ValueError('Choose a GRPO output directory separate from the input SFT run')
print('Input SFT adapter:', SFT_ADAPTER)
print('GRPO output on Drive:', GRPO)


## 5. GRPO continuation with trainable SFT LoRA
Generates fresh groups during training; no precomputed inference file is required.
mmBERT reward runs on CPU by default to preserve GPU VRAM; use reward.device=cuda if memory permits.
beta=0 explicitly avoids treating the pre-SFT base as the KL reference.


In [ ]:
grpo_args = {**common, 'model.adapter': SFT_ADAPTER, 'model.adapter_revision': ADAPTER_REVISION,
             'training.output_dir': GRPO}
if not RUN_FULL:
    grpo_args.update({'data.limit': 8, 'training.max_steps': 1, 'training.save_steps': 1})
run('mmcultureqa.training.grpo', 'configs/grpo.yaml', grpo_args)


In [ ]:
GRPO_FINAL = Path(GRPO) / 'final'
if not (GRPO_FINAL / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'GRPO did not produce a final adapter: {GRPO_FINAL}')
print('GRPO adapter saved on Drive:', GRPO_FINAL)
